# Causal estimator benchmark: mboostrs against EconML and DoubleML

A user who estimates a treatment effect from observational data needs to know how
far the estimate is from the truth at the library's default settings, how that error
compares with other libraries' estimators, and how long each takes. This study answers
those three questions for mboostrs' augmented inverse probability weighting estimator,
`mb.aipw`, against EconML, DoubleML and a hand-built R-learner, with the confounders
and draws held fixed. It also measures which `aipw` settings change the error.

In practice: at its default settings (`smooth=False`, `outcome_model="cv"`), `mb.aipw`
has a median absolute error of 434 dollars in the average treatment effect (ATE) over
ten draws whose true ATE is about 4,270 dollars, and its 90% intervals contain the
truth on 9 of the 10 draws. EconML's `LinearDML` with gradient-boosted nuisance
learners, and the R-learner with the same nuisances, have the lowest median error, 331
dollars. Among the `aipw` settings, turning off propensity clipping is the one that
produces errors of tens of thousands of dollars.

## Design

**Data.** The draws come from the job-training generator of the
[job-training analysis](causal_discovery.ipynb). The treatment is a binary indicator of
enrolment in a training program, the outcome is later earnings in dollars, and the
applicants' covariates are survey records from the `CPS1988` data of the R package AER.
The generator sets every applicant's individual effect, so the true ATE of each draw is
known: it is the mean individual effect over the draw's 12,000 training applicants.
There are twelve draws: ten with a planted effect (true ATE between 4,217 and 4,292
dollars) and two null draws with no effect. Earnings are missing for applicants who did
not answer the follow-up survey. Every arm drops those rows and weights each responder
by the inverse of its estimated probability of responding, from one logistic
regression on enrolment and the confounders; these response weights run from 1.045 to
1.654. The confounders are prior earnings, education, months unemployed, experience,
region and metropolitan residence.

**Terms.** A *nuisance learner* is a model the estimator fits for a quantity it needs
but does not report. Here there are two: the propensity, the probability of enrolment
given the confounders, and the outcome regression, the expected earnings given the
confounders and enrolment. All arms *cross-fit* them: each row's nuisance predictions
come from models fitted on the other four of five folds. HGB is scikit-learn's
histogram gradient boosting (`HistGradientBoostingRegressor` and `...Classifier`,
default settings). GAM is mboostrs' scikit-learn adapter (`GAMBoostRegressor` and
`GAMBoostClassifier`) with P-splines for the four continuous confounders, linear terms
for the four indicators, 100 boosting steps and step length 0.1.

**Arms.**

- **mboostrs `aipw`.** AIPW adds to the outcome-regression estimate of the effect an
  inverse-propensity-weighted mean of the outcome residuals; the estimate is consistent
  when either nuisance model is correct. `aipw` fits both nuisances by boosting inside
  mboostrs. Twelve configurations cross `smooth=False` (linear terms) and `smooth=True`
  (penalized splines for numeric confounders) with these outcome-model settings:
  `"cv"`, which fits the pooled, separate and two-part outcome models and keeps the one
  with the lowest out-of-fold squared error; `"pooled"`, one model with treatment as a
  linear term; and `"auto"`, the rule-based choice that was the default in mboostrs
  0.1.3, which fitted the pooled model on every draw here. The two-part model multiplies
  a model for the probability of positive earnings by a model for log earnings among
  the positive values. With `"auto"`, three more configurations each change one
  setting: no propensity clipping (`propensity_clip=None`; the default clips to 0.01 and
  0.99), no tuning (`tune=False`, a fixed 200 boosting steps in place of inner
  cross-validation), and `propensity_learner="smooth+trees"`. The default of
  `aipw` is `smooth=False` with `outcome_model="cv"`.
- **mboostrs `effect_model`**, with `terms="auto"` and `terms="smooth"`. It models the
  individual effect as a function of the confounders; its ATE is the mean fitted
  individual effect, with no interval.
- **EconML `LinearDML` and `CausalForestDML`**, each with HGB or GAM nuisances. Both
  subtract the cross-fitted nuisance predictions from the outcome and the treatment and
  then regress the outcome residual on the treatment residual: `LinearDML` with an effect
  linear in the confounders, `CausalForestDML` with a random forest. The forest gives no
  ATE interval.
- **DoubleML `IRM`** (interactive regression model), with HGB or GAM nuisances. It uses
  the AIPW score with propensities trimmed at 0.01 and 0.99.
- **Hand-built R-learner**, with linear (statsmodels), HGB or GAM nuisances. It regresses
  the outcome residual on the treatment residual and its products with the centred
  confounders by weighted least squares; the coefficient of the treatment residual is the
  ATE. With the same nuisances and folds it computes the same estimate as `LinearDML`.

**Folds.** Five folds, seeded `101 + k` for draw `k`. EconML, DoubleML and the R-learner
share one partition. The mboostrs arms build their own stratified partition from the same
seed, so their folds contain different rows.

**Measures.** The absolute ATE error is the absolute difference between the estimate
and the true ATE, in dollars; lower is better. Each arm is summarised by the median and
the largest absolute error over the ten effect draws, the share of those draws whose 90%
interval contains the true ATE (nominal 0.9), and the median wall time of fit and
prediction on a shared 10-core Apple Silicon laptop. On the two null draws, a 90%
interval that excludes zero is a false rejection.

**Measurements.** All arms were measured with mboostrs 0.1.3 (development builds of the
repository), EconML 0.17.0, DoubleML 0.11.4, scikit-learn 1.9.0 and statsmodels 0.15.0.
The twelve `aipw` configurations were measured together on the ten effect draws; the
other arms were measured in a separate run on all twelve draws. Both runs used the same
draws, folds and seeds. The protocol, the arms of the library comparison and its
contrasts were fixed before the first arm was fitted.

In [1]:
import json
from pathlib import Path

import polars as pl
from great_tables import GT
from IPython.display import display

# This page runs from docs/book/benchmarks; the records sit beside the protocol.
FOLDER = Path("causal_estimators")
lines = (FOLDER / "per_draw_results.jsonl").read_text().splitlines()
keep = ["draw", "arm", "null", "ate_hat", "ate_error", "ate_cover90", "ate_reject90", "wall_s", "error"]
library_draws = pl.DataFrame([{k: json.loads(line).get(k) for k in keep} for line in lines],
                             infer_schema_length=None)

aipw_file = json.loads((FOLDER / "rerun_aipw_2026-10-23.json").read_text())
truth = pl.col("draw").replace_strict(aipw_file["truth_by_draw"], return_dtype=pl.Float64)
aipw_draws = pl.DataFrame(aipw_file["rows"], infer_schema_length=None).with_columns(
    (pl.col("ate_hat") - truth).alias("ate_error"),
    ((pl.col("lo") <= truth) & (pl.col("hi") >= truth)).alias("ate_cover90"),
)

# The pooled AIPW configuration was fitted in both measurements. Identical estimates
# show that the two measurements used the same draws, folds and seeds.
both = library_draws.filter(pl.col("arm") == "mb_aipw_pooled").join(
    aipw_draws.filter(pl.col("arm") == "smooth_False__pooled"), on="draw", suffix="_aipw")
print(f"AIPW configurations: {aipw_draws['arm'].n_unique()}, {aipw_draws.height} records, "
      f"mboostrs {aipw_file['mboostrs']}")
print(f"Other arms: {library_draws.height} records, "
      f"{library_draws['error'].is_not_null().sum()} failed calls")
print("Largest difference between the two pooled-configuration estimates: "
      f"{(both['ate_hat'] - both['ate_hat_aipw']).abs().max():.6f} dollars")

AIPW configurations: 12, 120 records, mboostrs 0.1.3
Other arms: 228 records, 0 failed calls
Largest difference between the two pooled-configuration estimates: 0.000000 dollars


The pooled `aipw` configuration was fitted in both runs and gives the same estimate on
every draw, which confirms that the runs share draws, folds and seeds. No call failed.

## AIPW configurations

In [2]:
def other_setting(settings):
    other = {k: v for k, v in settings.items() if k not in ("smooth", "outcome_model")}
    return ", ".join(f"{k}={v!r}" for k, v in other.items()) or "none"

fitted = (aipw_draws.group_by("arm", "outcome_model").len().sort("len", descending=True)
          .group_by("arm", maintain_order=True)
          .agg(pl.format("{} ({})", "outcome_model", "len").str.join(", ").alias("Outcome model fitted (draws)")))
config = (
    aipw_draws.group_by("arm").agg(
        pl.col("ate_error").abs().median().alias("Median absolute error ($)"),
        pl.col("ate_error").abs().max().alias("Largest absolute error ($)"),
        pl.col("ate_cover90").mean().alias("90% coverage"),
        pl.col("seconds").median().alias("Median seconds"),
    )
    .join(fitted, on="arm")
    .with_columns(
        pl.col("arm").map_elements(lambda a: str(aipw_file["summary"][a]["settings"]["smooth"]), return_dtype=pl.String).alias("smooth"),
        pl.col("arm").map_elements(lambda a: aipw_file["summary"][a]["settings"].get("outcome_model", "auto"), return_dtype=pl.String).alias("outcome_model"),
        pl.col("arm").map_elements(lambda a: other_setting(aipw_file["summary"][a]["settings"]), return_dtype=pl.String).alias("Other setting"),
    )
    .sort("Median absolute error ($)")
    .select("smooth", "outcome_model", "Other setting", pl.exclude("arm", "smooth", "outcome_model", "Other setting"))
)
(
    GT(config)
    .tab_header(title="mb.aipw configurations on the ten effect draws",
                subtitle="Absolute ATE error in dollars, lower is better; true ATE about 4,270 dollars; measured with mboostrs 0.1.3")
    .fmt_number(columns=["Median absolute error ($)", "Largest absolute error ($)"], decimals=0)
    .fmt_number(columns=["90% coverage"], decimals=1)
    .fmt_number(columns=["Median seconds"], decimals=2)
    .tab_source_note("The aipw default is smooth=False with outcome_model='cv'. outcome_model='auto' was the default in 0.1.3; "
                     "the three single-setting variants were measured with it.")
)

GT(_tbl_data=shape: (12, 8)
┌────────┬────────────┬────────────┬────────────┬────────────┬────────────┬────────────┬───────────┐
│ smooth ┆ outcome_mo ┆ Other      ┆ Median     ┆ Largest    ┆ 90%        ┆ Median     ┆ Outcome   │
│ ---    ┆ del        ┆ setting    ┆ absolute   ┆ absolute   ┆ coverage   ┆ seconds    ┆ model     │
│ str    ┆ ---        ┆ ---        ┆ error ($)  ┆ error ($)  ┆ ---        ┆ ---        ┆ fitted    │
│        ┆ str        ┆ str        ┆ ---        ┆ ---        ┆ f64        ┆ f64        ┆ (draws)   │
│        ┆            ┆            ┆ f64        ┆ f64        ┆            ┆            ┆ ---       │
│        ┆            ┆            ┆            ┆            ┆            ┆            ┆ str       │
╞════════╪════════════╪════════════╪════════════╪════════════╪════════════╪════════════╪═══════════╡
│ False  ┆ cv         ┆ none       ┆ 434.056651 ┆ 2321.76054 ┆ 0.9        ┆ 2.673816   ┆ two_part  │
│        ┆            ┆            ┆            ┆ 7          ┆            ┆            ┆ (9),      │
│        ┆            ┆            ┆            ┆            ┆            ┆            ┆ pooled    │
│        ┆            ┆            ┆            ┆            ┆            ┆            ┆ (1)       │
│ True   ┆ cv         ┆ none       ┆ 550.077499 ┆ 1705.73721 ┆ 0.9        ┆ 8.047832   ┆ two_part  │
│        ┆            ┆            ┆            ┆ 9          ┆            ┆            ┆ (9),      │
│        ┆            ┆            ┆            ┆            ┆            ┆            ┆ separate  │
│        ┆            ┆            ┆            ┆            ┆            ┆            ┆ (1)       │
│ True   ┆ pooled     ┆ none       ┆ 714.296337 ┆ 2127.86708 ┆ 0.8        ┆ 5.538578   ┆ pooled    │
│        ┆            ┆            ┆            ┆ 7          ┆            ┆            ┆ (10)      │
│ False  ┆ pooled     ┆ none       ┆ 749.270171 ┆ 2043.54765 ┆ 0.6        ┆ 1.288734   ┆ pooled    │
│        ┆            ┆            ┆            ┆ 7          ┆            ┆            ┆ (10)      │
│ False  ┆ auto       ┆ none       ┆ 790.307736 ┆ 2321.76054 ┆ 0.6        ┆ 1.520275   ┆ pooled    │
│        ┆            ┆            ┆            ┆ 7          ┆            ┆            ┆ (10)      │
│ …      ┆ …          ┆ …          ┆ …          ┆ …          ┆ …          ┆ …          ┆ …         │
│ True   ┆ auto       ┆ none       ┆ 1137.07181 ┆ 8205.07889 ┆ 0.6        ┆ 5.868563   ┆ pooled    │
│        ┆            ┆            ┆ 6          ┆ 1          ┆            ┆            ┆ (10)      │
│ False  ┆ auto       ┆ propensity ┆ 1223.80612 ┆ 2193.26351 ┆ 0.6        ┆ 3.069885   ┆ pooled    │
│        ┆            ┆ _learner=' ┆            ┆ 5          ┆            ┆            ┆ (10)      │
│        ┆            ┆ smooth+tre ┆            ┆            ┆            ┆            ┆           │
│        ┆            ┆ …          ┆            ┆            ┆            ┆            ┆           │
│ False  ┆ auto       ┆ propensity ┆ 1531.58465 ┆ 59688.5384 ┆ 0.8        ┆ 1.507143   ┆ pooled    │
│        ┆            ┆ _clip=None ┆ 5          ┆ 16         ┆            ┆            ┆ (10)      │
│ True   ┆ auto       ┆ propensity ┆ 1675.05275 ┆ 7711.23071 ┆ 0.6        ┆ 3.69787    ┆ pooled    │
│        ┆            ┆ _learner=' ┆ 5          ┆ 8          ┆            ┆            ┆ (10)      │
│        ┆            ┆ smooth+tre ┆            ┆            ┆            ┆            ┆           │
│        ┆            ┆ …          ┆            ┆            ┆            ┆            ┆           │
│ True   ┆ auto       ┆ propensity ┆ 8646.35045 ┆ 87545.6476 ┆ 0.8        ┆ 5.948253   ┆ pooled    │
│        ┆            ┆ _clip=None ┆ 3          ┆ 43         ┆            ┆            ┆ (10)      │
└────────┴────────────┴────────────┴────────────┴────────────┴────────────┴────────────┴───────────┘, _body=<great_tables._gt_data.Body object at 0x109217d70>, _boxhead=Boxhead([ColInfo(var='smooth', type=<ColInfoTypeEnum.default: 1>, 

- `outcome_model="cv"` gives the lowest median error at both `smooth` settings, 434
  dollars with linear nuisances and 550 with smooth ones, and the only coverage of 0.9.
  It chose the two-part outcome model on 9 of the 10 draws.
- With `outcome_model="auto"`, which chose the pooled model, coverage is 0.6 and the
  median error is 790 (linear) and 1,137 dollars (smooth).
- `smooth=True` costs about three times the fit time of `smooth=False` (8.0 against 2.7
  seconds with `"cv"`) and does not lower the median error on this design.
- Removing propensity clipping gives a largest error of 59,689 dollars with linear
  nuisances and 87,546 with smooth ones. The smallest cross-fitted propensity of a draw is below
  1e-15 on every draw with linear nuisances and below 1e-3 with smooth ones. AIPW
  divides by the propensity, so without clipping a few rows receive very large weights.
- `tune=False` is the fastest configuration (0.29 seconds with linear nuisances), with a
  median error of 878 dollars and coverage 0.5.

## Default AIPW and the other estimators

In [3]:
LABELS = {
    "ec_lin_hgb": "EconML LinearDML, HGB nuisances", "ec_lin_gam": "EconML LinearDML, GAM nuisances",
    "ec_cf_hgb": "EconML CausalForestDML, HGB nuisances", "ec_cf_gam": "EconML CausalForestDML, GAM nuisances",
    "dml_irm_hgb": "DoubleML IRM, HGB nuisances", "dml_irm_gam": "DoubleML IRM, GAM nuisances",
    "hand_lin": "R-learner, linear nuisances", "hand_hgb": "R-learner, HGB nuisances",
    "hand_gam": "R-learner, GAM nuisances", "mb_eff_auto": "mboostrs effect_model, terms='auto'",
    "mb_eff_smooth": "mboostrs effect_model, terms='smooth'"}
DEFAULT = "smooth_False__outcome_cv"
measures = [
    pl.col("ate_error").abs().median().alias("Median absolute error ($)"),
    pl.col("ate_error").abs().max().alias("Largest absolute error ($)"),
    pl.col("ate_cover90").cast(pl.Float64).mean().alias("90% coverage"),
]
others = library_draws.filter(pl.col("arm").is_in(list(LABELS)))
null_rejections = others.filter(pl.col("null")).group_by("arm").agg(
    pl.when(pl.col("ate_reject90").is_null().all()).then(pl.lit("no interval"))
    .otherwise(pl.format("{} of {}", pl.col("ate_reject90").sum(), pl.len())).alias("Null draws: zero excluded"))
other_rows = (others.filter(~pl.col("null")).group_by("arm")
              .agg(*measures, pl.col("wall_s").median().alias("Median seconds")).join(null_rejections, on="arm"))
default_row = (aipw_draws.filter(pl.col("arm") == DEFAULT).group_by("arm")
               .agg(*measures, pl.col("seconds").median().alias("Median seconds"))
               .with_columns(pl.lit("not measured").alias("Null draws: zero excluded")))
labels = {**LABELS, DEFAULT: "mboostrs aipw, default settings"}
comparison = (pl.concat([default_row, other_rows]).sort("Median absolute error ($)")
              .with_columns(pl.col("arm").replace_strict(labels).alias("Arm")).drop("arm").select("Arm", pl.exclude("Arm")))
(
    GT(comparison)
    .tab_header(title="Default mb.aipw against the other estimators on the ten effect draws",
                subtitle="Absolute ATE error in dollars, lower is better; coverage of the 90% interval, nominal 0.9")
    .fmt_number(columns=["Median absolute error ($)", "Largest absolute error ($)"], decimals=0)
    .fmt_number(columns=["90% coverage"], decimals=1).fmt_number(columns=["Median seconds"], decimals=2)
    .sub_missing(missing_text="no interval")
    .tab_source_note("Null draws: how many of the two no-effect draws had a 90% interval that excludes zero (lower is better). "
                     "The default aipw row comes from the AIPW measurement, which has no null draws.")
)

GT(_tbl_data=shape: (12, 6)
┌─────────────────┬─────────────────┬─────────────────┬──────────────┬───────────┬─────────────────┐
│ Arm             ┆ Median absolute ┆ Largest         ┆ 90% coverage ┆ Median    ┆ Null draws:     │
│ ---             ┆ error ($)       ┆ absolute error  ┆ ---          ┆ seconds   ┆ zero excluded   │
│ str             ┆ ---             ┆ ($)             ┆ f64          ┆ ---       ┆ ---             │
│                 ┆ f64             ┆ ---             ┆              ┆ f64       ┆ str             │
│                 ┆                 ┆ f64             ┆              ┆           ┆                 │
╞═════════════════╪═════════════════╪═════════════════╪══════════════╪═══════════╪═════════════════╡
│ R-learner, HGB  ┆ 330.566282      ┆ 1762.60792      ┆ 0.9          ┆ 6.747687  ┆ 1 of 2          │
│ nuisances       ┆                 ┆                 ┆              ┆           ┆                 │
│ EconML          ┆ 330.566282      ┆ 1762.60792      ┆ 0.9          ┆ 9.974171  ┆ 1 of 2          │
│ LinearDML, HGB  ┆                 ┆                 ┆              ┆           ┆                 │
│ nuisance…       ┆                 ┆                 ┆              ┆           ┆                 │
│ mboostrs aipw,  ┆ 434.056651      ┆ 2321.760547     ┆ 0.9          ┆ 2.673816  ┆ not measured    │
│ default         ┆                 ┆                 ┆              ┆           ┆                 │
│ setting…        ┆                 ┆                 ┆              ┆           ┆                 │
│ R-learner, GAM  ┆ 469.328101      ┆ 5951.665003     ┆ 0.9          ┆ 0.275919  ┆ 0 of 2          │
│ nuisances       ┆                 ┆                 ┆              ┆           ┆                 │
│ EconML          ┆ 469.328101      ┆ 5951.665003     ┆ 0.9          ┆ 0.314682  ┆ 0 of 2          │
│ LinearDML, GAM  ┆                 ┆                 ┆              ┆           ┆                 │
│ nuisance…       ┆                 ┆                 ┆              ┆           ┆                 │
│ …               ┆ …               ┆ …               ┆ …            ┆ …         ┆ …               │
│ mboostrs        ┆ 877.767664      ┆ 4648.988216     ┆ null         ┆ 50.115451 ┆ no interval     │
│ effect_model,   ┆                 ┆                 ┆              ┆           ┆                 │
│ terms='…        ┆                 ┆                 ┆              ┆           ┆                 │
│ DoubleML IRM,   ┆ 1693.023522     ┆ 4593.747599     ┆ 0.6          ┆ 10.65834  ┆ 0 of 2          │
│ HGB nuisances   ┆                 ┆                 ┆              ┆           ┆                 │
│ DoubleML IRM,   ┆ 2067.265494     ┆ 7621.72746      ┆ 0.2          ┆ 0.36561   ┆ 1 of 2          │
│ GAM nuisances   ┆                 ┆                 ┆              ┆           ┆                 │
│ EconML CausalFo ┆ 2915.462142     ┆ 46270.673908    ┆ null         ┆ 1.169814  ┆ no interval     │
│ restDML, GAM    ┆                 ┆                 ┆              ┆           ┆                 │
│ nu…             ┆                 ┆                 ┆              ┆           ┆                 │
│ EconML CausalFo ┆ 3206.248434     ┆ 80010.54338     ┆ null         ┆ 10.265148 ┆ no interval     │
│ restDML, HGB    ┆                 ┆                 ┆              ┆           ┆                 │
│ nu…             ┆                 ┆                 ┆              ┆           ┆                 │
└─────────────────┴─────────────────┴─────────────────┴──────────────┴───────────┴─────────────────┘, _body=<great_tables._gt_data.Body object at 0x10947ee70>, _boxhead=Boxhead([ColInfo(var='Arm', type=<ColInfoTypeEnum.default: 1>, column_label='Arm', column_align='left', column_width=None, stub_level=None), ColInfo(var='Median absolute error ($)', type=<ColInfoTypeEnum.default: 1>, column_label='Median absolute error ($)', column_align='right', column_width=None, stub_level=None), ColInfo(var='Largest absolute error ($)', type=<Col

- EconML `LinearDML` and the R-learner give identical estimates when their nuisances
  are the same, as the shared design predicts. With HGB nuisances they have the lowest
  median error, 331 dollars, and a largest error of 1,763 dollars.
- The default `aipw` has a median error of 434 dollars and a largest error of 2,322, with
  coverage 0.9. Its median time, 2.7 seconds, is below the 10.0 seconds of `LinearDML`
  with HGB nuisances; the two times come from separate runs on a shared machine.
- GAM nuisances give `LinearDML` a median error of 469 dollars, about 140 above HGB
  nuisances, a larger worst draw (5,952 against 1,763 dollars), and a run about 30
  times faster (0.31 against 10.0 seconds).
- DoubleML `IRM` and the causal forests have median errors between 1,693 and 3,206
  dollars. `IRM` with GAM nuisances covers the truth on 2 of 10 draws.
- `effect_model` has median errors of 626 (`terms="smooth"`) and 878 dollars
  (`terms="auto"`) and is the slowest arm, at about 50 to 60 seconds.

## Contrasts between arms

The protocol fixes contrasts in three groups: the same framework with different
nuisance learners (a), different frameworks with the same nuisances (b), and a linear
against a forest or smooth second stage (c). Each contrast is the first arm minus the
second, for the ATE bias (the mean signed error), the ATE root mean squared error
(RMSE), the 90% coverage, and, for arms that estimate individual effects, the median
PEHE. PEHE is the root mean squared error of the estimated individual effects against
the true ones, over the test applicants whose true enrolment probability is between
0.02 and 0.98. Lower RMSE and PEHE are better. The group (b) rows for the default `aipw`
are computed from the two runs' per-draw records.

In [4]:
results = json.loads((FOLDER / "results.json").read_text())
# Contrasts whose arms ran in the library-comparison measurement and do not involve AIPW.
recorded = [
    {"Group": c["group"], "Contrast": c["label"], "ATE bias ($)": c["ate_bias"],
     "ATE RMSE ($)": c["ate_rmse"], "90% coverage": c["ate_cover90"], "Median PEHE ($)": c["pehe_pool_median"]}
    for c in results["contrasts"] if not c["a"].startswith("mb_aipw") and not c["b"].startswith("mb_aipw")
]
def ate_summary(frame):
    effect = frame.filter(~pl.col("draw").str.starts_with("Z"))
    return {"bias": effect["ate_error"].mean(), "rmse": (effect["ate_error"] ** 2).mean() ** 0.5,
            "cover": effect["ate_cover90"].cast(pl.Float64).mean()}
default_summary = ate_summary(aipw_draws.filter(pl.col("arm") == DEFAULT))
for arm in ["ec_lin_hgb", "ec_lin_gam", "dml_irm_hgb", "dml_irm_gam"]:
    other = ate_summary(library_draws.filter(pl.col("arm") == arm))
    recorded.append({"Group": "b_framework", "Contrast": f"Default mboostrs AIPW minus {LABELS[arm]}",
                     "ATE bias ($)": default_summary["bias"] - other["bias"],
                     "ATE RMSE ($)": default_summary["rmse"] - other["rmse"],
                     "90% coverage": default_summary["cover"] - other["cover"], "Median PEHE ($)": None})
GROUPS = {"a_nuisance": "a. Nuisance learners", "b_framework": "b. Estimation framework", "c_second_stage": "c. Second stage"}
contrast_table = (pl.DataFrame(recorded, infer_schema_length=None).sort(["Group", "Contrast"])
                  .with_columns(pl.col("Group").replace_strict(GROUPS)))
(
    GT(contrast_table, groupname_col="Group")
    .tab_header(title="Contrasts between arms on the ten effect draws",
                subtitle="Each row is the first arm minus the second; a negative ATE RMSE or PEHE difference favours the first arm")
    .fmt_number(columns=["ATE bias ($)", "ATE RMSE ($)", "Median PEHE ($)"], decimals=0)
    .fmt_number(columns=["90% coverage"], decimals=1)
    .sub_missing(missing_text="not available")
    .tab_source_note("PEHE compares individual-effect estimates and exists only for arms that estimate them.")
)

GT(_tbl_data=shape: (13, 6)
┌─────────────────┬─────────────────┬──────────────┬──────────────┬──────────────┬─────────────────┐
│ Group           ┆ Contrast        ┆ ATE bias ($) ┆ ATE RMSE ($) ┆ 90% coverage ┆ Median PEHE ($) │
│ ---             ┆ ---             ┆ ---          ┆ ---          ┆ ---          ┆ ---             │
│ str             ┆ str             ┆ f64          ┆ f64          ┆ f64          ┆ f64             │
╞═════════════════╪═════════════════╪══════════════╪══════════════╪══════════════╪═════════════════╡
│ a. Nuisance     ┆ DoubleML IRM:   ┆ 1416.453644  ┆ 398.324046   ┆ -0.4         ┆ null            │
│ learners        ┆ GAM minus HGB   ┆              ┆              ┆              ┆                 │
│                 ┆ nu…             ┆              ┆              ┆              ┆                 │
│ a. Nuisance     ┆ EconML          ┆ -707.200073  ┆ 1419.66271   ┆ 0.0          ┆ 27.347573       │
│ learners        ┆ LinearDML: GAM  ┆              ┆              ┆              ┆                 │
│                 ┆ minus HG…       ┆              ┆              ┆              ┆                 │
│ a. Nuisance     ┆ Hand R-learner: ┆ -707.200073  ┆ 1419.66271   ┆ 0.0          ┆ 27.347573       │
│ learners        ┆ GAM minus HGB … ┆              ┆              ┆              ┆                 │
│ a. Nuisance     ┆ Hand R-learner: ┆ -617.630787  ┆ 1319.040015  ┆ 0.2          ┆ 88.639106       │
│ learners        ┆ GAM minus stat… ┆              ┆              ┆              ┆                 │
│ b. Estimation   ┆ Default         ┆ -520.286196  ┆ -2095.105811 ┆ 0.7          ┆ null            │
│ framework       ┆ mboostrs AIPW   ┆              ┆              ┆              ┆                 │
│                 ┆ minus Do…       ┆              ┆              ┆              ┆                 │
│ …               ┆ …               ┆ …            ┆ …            ┆ …            ┆ …               │
│ b. Estimation   ┆ Same GAM        ┆ -2.5466e-11  ┆ 3.6380e-11   ┆ 0.0          ┆ -1.8190e-12     │
│ framework       ┆ nuisances,      ┆              ┆              ┆              ┆                 │
│                 ┆ linear sec…     ┆              ┆              ┆              ┆                 │
│ b. Estimation   ┆ Same HGB        ┆ -1.2790e-11  ┆ 1.3870e-11   ┆ 0.0          ┆ -9.0949e-13     │
│ framework       ┆ nuisances:      ┆              ┆              ┆              ┆                 │
│                 ┆ LinearDML …     ┆              ┆              ┆              ┆                 │
│ c. Second stage ┆ GAM nuisances:  ┆ -1766.591789 ┆ 12797.543513 ┆ null         ┆ 1001.988757     │
│                 ┆ forest minus    ┆              ┆              ┆              ┆                 │
│                 ┆ li…             ┆              ┆              ┆              ┆                 │
│ c. Second stage ┆ HGB nuisances:  ┆ -6896.653875 ┆ 24817.867506 ┆ null         ┆ 1001.609823     │
│                 ┆ forest minus    ┆              ┆              ┆              ┆                 │
│                 ┆ li…             ┆              ┆              ┆              ┆                 │
│ c. Second stage ┆ mboostrs        ┆ 1187.127816  ┆ -408.098327  ┆ null         ┆ 400.355156      │
│                 ┆ nuisances:      ┆              ┆              ┆              ┆                 │
│                 ┆ smooth min…     ┆              ┆              ┆              ┆                 │
└─────────────────┴─────────────────┴──────────────┴──────────────┴──────────────┴─────────────────┘, _body=<great_tables._gt_data.Body object at 0x10947d640>, _boxhead=Boxhead([ColInfo(var='Group', type=<ColInfoTypeEnum.row_group: 3>, column_label='Group', column_align='left', column_width=None, stub_level=None), ColInfo(var='Contrast', type=<ColInfoTypeEnum.default: 1>, column_label='Contrast', column_align='left', column_width=None, stub_level=None), ColInfo(var='ATE bias ($)', type=<ColInfoTypeEnum.default: 1>, column_label='ATE

- The default `aipw` has a larger ATE RMSE than `LinearDML` with HGB nuisances, by 223
  dollars, and a smaller one than `LinearDML` with GAM nuisances and both `IRM` arms,
  by 1,197 to 2,095 dollars. Its bias is positive (134 dollars); the linear arms' biases
  are negative.
- Forest second stages raise the ATE RMSE by about 12,800 (GAM) and 24,800 dollars
  (HGB) and the median PEHE by about 1,000 dollars over the linear second stage.
- The study applies no significance test. With ten effect draws, the contrasts are
  descriptive magnitudes.

## What the results mean for a user

- The default `aipw` settings give the lowest median error of the twelve
  configurations measured on this design and, with `smooth=True, outcome_model="cv"`,
  the only coverage of 0.9.
- Keep the default propensity clipping. On data where some estimated propensities are
  near 0 or 1, `propensity_clip=None` can move the ATE by tens of thousands of dollars.
- On this design EconML `LinearDML` with HGB nuisances has a lower error than the default
  `aipw`, by about 100 dollars at the median. Ten draws do not establish whether that
  difference persists.

## Limits

- **Ten effect draws and two null draws, one design.** The medians and extremes are
  descriptive; no significance test is applied. The design has 12,000 training rows and
  one confounder set.
- **Null draws not measured for the `aipw` configurations.** The false-rejection rate of
  the default `aipw` is unknown.
- **Library version.** The results were measured with mboostrs 0.1.3 development builds.
  The `aipw` variants without clipping, without tuning and with tree propensities were
  measured with `outcome_model="auto"`, so they show the effect of each setting
  relative to that configuration, not to the default `"cv"`. The arms that use the GAM
  adapter and the `effect_model` arms were measured in the other run and not repeated.
- **Different partitions for mboostrs and the other arms.** Their folds contain
  different rows.
- **DoubleML's response weights.** In DoubleML 0.11.4 the response weights multiply the
  ATE score only; its nuisance learners are fitted without them.
- **Timings.** They come from a shared machine under varying load and from two runs, so
  they compare arms roughly.

## Reproduction

The files are in `docs/book/benchmarks/causal_estimators/`: `PROTOCOL.md` (draws,
weights, the exact call of every arm, measures, contrasts), `per_draw_results.jsonl`
(the run of the other arms, one record per draw and arm), `results.json` (its
summaries and contrasts) and `rerun_aipw_2026-10-23.json` (the `aipw` configurations).
The scripts that generated the records are not in the repository. The applicant
covariates are `AER::CPS1988`; the protocol gives the file's sha256 and the seeds.